# Deep Learning Training with PRE-SCALED CIC-IDS-2018 Dataset

This notebook trains DL models (CNN, LSTM) using data that has ALREADY been scaled with PowerTransformer.

**Key Changes from Original:**
1. Load pre-scaled CSV instead of raw data
2. Remove StandardScaler (data already scaled)
3. Keep all other training logic the same

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings
warnings.filterwarnings('ignore')
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import (Dense, Dropout, Conv1D, MaxPooling1D, Flatten,
                          LSTM, BatchNormalization)
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("🚀 DEEP LEARNING TRAINING WITH PRE-SCALED DATASET")
print("="*80)

## 2. Configuration

In [ ]:
INPUT_FILE = "archive/cicids2018_scaled.csv"  # Pre-scaled data
SCALER_FILE = "archive/fitted_scalers.pkl"  # For reference/test data

BASE_MODEL_DIR = "trained_models/dl/"

# Data loading settings
CHUNK_SIZE = 100000
MAX_ROWS = None  # None = load all data
TEST_SIZE = 0.2
VALIDATION_SPLIT = 0.2
RANDOM_STATE = 42
SAVE_MODELS = True

# Training parameters
EPOCHS = 50
BATCH_SIZE = 256
PATIENCE = 10

# Per-model dataset size control
MODEL_SAMPLE_SIZES = {
    'CNN': None,      # Use full dataset
    'LSTM': 200000    # LSTM is slower, limit to 200k rows
}

## 3. Load Pre-Scaled Dataset

In [ ]:
print(f"\n📂 Loading PRE-SCALED data from: {INPUT_FILE}")
print("⏳ Reading data in chunks...\n")

chunks = []
total_rows = 0

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    if MAX_ROWS and total_rows >= MAX_ROWS:
        break
    
    if MAX_ROWS and total_rows + len(chunk) > MAX_ROWS:
        chunk = chunk.head(MAX_ROWS - total_rows)
    
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"   Chunk {i+1}: {len(chunk):,} rows | Total: {total_rows:,}")

df_full = pd.concat(chunks, ignore_index=True)
del chunks

print(f"\n✅ Dataset loaded successfully!")
print(f"   Total rows: {len(df_full):,}")
print(f"   Total columns: {len(df_full.columns)}")

## 4. Verify Data is Pre-Scaled

In [ ]:
print("\n" + "="*80)
print("🔍 VERIFYING DATA IS PRE-SCALED")
print("="*80)

sample_features = df_full.select_dtypes(include=[np.number]).columns[:5]
print(f"\n📊 Sample feature statistics (should be mean≈0, std≈1):")
print(f"   {'Feature':<30} {'Mean':>10} {'Std':>10}")
print(f"   {'-'*52}")

for col in sample_features:
    if col != 'Label_Binary':
        mean = df_full[col].mean()
        std = df_full[col].std()
        print(f"   {col:<30} {mean:>10.4f} {std:>10.4f}")

print(f"\n✅ Data appears to be pre-scaled!")

## 5. Prepare Dataset

In [ ]:
print("\n" + "="*80)
print("🔧 PREPARING DATA FOR TRAINING")
print("="*80)

# Separate features and labels
X_full = df_full.drop(['Label_Binary'], axis=1)
y_full = df_full['Label_Binary']

X_full = X_full.select_dtypes(include=[np.number])

print(f"\n✅ Features shape: {X_full.shape}")
print(f"✅ Labels shape: {y_full.shape}")
print(f"\nNumber of features: {X_full.shape[1]}")
print(f"Class distribution:")
print(y_full.value_counts())

# Store feature count for model building
NUM_FEATURES = X_full.shape[1]

## 6. Model Builders

In [ ]:
def build_cnn_model(input_shape):
    """
    Build 1D CNN model for network traffic classification.
    Input data is already scaled, no need for normalization layers.
    """
    model = Sequential([
        # Reshape for Conv1D: (batch, features, 1)
        tf.keras.layers.Reshape((input_shape[0], 1), input_shape=input_shape),
        
        # Conv Block 1
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        
        # Conv Block 2
        Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        
        # Conv Block 3
        Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.4),
        
        # Dense layers
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

def build_lstm_model(input_shape):
    """
    Build LSTM model for network traffic classification.
    Input data is already scaled.
    """
    model = Sequential([
        # Reshape for LSTM: (batch, timesteps, features)
        tf.keras.layers.Reshape((input_shape[0], 1), input_shape=input_shape),
        
        # LSTM layers
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        
        # Dense layers
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

## 7. Helper Function: Sample Data for Model

In [ ]:
def get_model_data(X_full, y_full, model_name, sample_size, test_size, random_state):
    """
    Get train/test split for a specific model with optional sampling.
    Returns data ready for DL models.
    
    NOTE: No scaling is applied here since data is PRE-SCALED!
    """
    
    if sample_size is None:
        X = X_full
        y = y_full
        print(f"📊 Using FULL dataset: {len(X):,} rows")
    else:
        print(f"📊 Sampling {sample_size:,} rows from {len(X_full):,} total rows")
        X, _, y, _ = train_test_split(
            X_full, y_full,
            train_size=sample_size,
            random_state=random_state,
            stratify=y_full
        )
        print(f"   Sampled class distribution: {y.value_counts().to_dict()}")
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    
    print(f"   Train set: {X_train.shape[0]:,} rows")
    print(f"   Test set:  {X_test.shape[0]:,} rows")
    
    # Convert to numpy arrays for Keras
    X_train = X_train.values.astype('float32')
    X_test = X_test.values.astype('float32')
    y_train = y_train.values.astype('float32')
    y_test = y_test.values.astype('float32')
    
    # ⚠️ NO SCALING HERE - Data is already scaled!
    
    return X_train, X_test, y_train, y_test

## 8. Train All Models

In [ ]:
model_builders = {
    'CNN': build_cnn_model,
    'LSTM': build_lstm_model
}

results = {}
histories = {}

for model_name, builder_func in model_builders.items():
    print("\n" + "="*80)
    print(f"🚀 TRAINING: {model_name}")
    print("="*80)
    
    # Create model-specific directory
    model_dir = os.path.join(BASE_MODEL_DIR, model_name.lower(), '')
    os.makedirs(model_dir, exist_ok=True)
    print(f"📁 Model directory: {model_dir}")
    
    # Get sample size for this model
    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)
    
    # Prepare data (NO SCALING - already scaled!)
    X_train, X_test, y_train, y_test = get_model_data(
        X_full, y_full,
        model_name,
        sample_size,
        TEST_SIZE,
        RANDOM_STATE
    )
    
    # Build model
    print(f"\n🏗️  Building {model_name} model...")
    model = builder_func(input_shape=(NUM_FEATURES,))
    
    print(f"\n📊 Model Summary:")
    model.summary()
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(model_dir, 'best_model.h5'),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Train model
    print(f"\n⏱️  Training {model_name}...")
    print(f"   Epochs: {EPOCHS}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Validation split: {VALIDATION_SPLIT}")
    
    start_time = time.time()
    
    history = model.fit(
        X_train, y_train,
        validation_split=VALIDATION_SPLIT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    
    train_time = time.time() - start_time
    print(f"\n✅ Training completed in {train_time/60:.2f} minutes")
    
    # Evaluate on test set
    print(f"\n🔮 Evaluating on test set...")
    test_loss, test_accuracy, test_precision, test_recall = model.evaluate(
        X_test, y_test, 
        batch_size=BATCH_SIZE,
        verbose=1
    )
    
    # Make predictions for F1-score
    y_pred_proba = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = (y_pred_proba > 0.5).astype(int).flatten()
    
    # Calculate F1-score
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    print(f"\n📊 Test Set Performance:")
    print(f"   Loss:      {test_loss:.4f}")
    print(f"   Accuracy:  {test_accuracy:.4f}")
    print(f"   Precision: {test_precision:.4f}")
    print(f"   Recall:    {test_recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    
    # Store results
    results[model_name] = {
        'accuracy': test_accuracy,
        'precision': test_precision,
        'recall': test_recall,
        'f1': f1,
        'loss': test_loss,
        'train_time': train_time,
        'sample_size': len(X_train)
    }
    
    histories[model_name] = history
    
    # Save model
    if SAVE_MODELS:
        print(f"\n💾 Saving model...")
        
        # Save model
        model_path = os.path.join(model_dir, "final_model.h5")
        model.save(model_path)
        
        # Save metadata
        metadata = {
            'model_name': model_name,
            'num_features': NUM_FEATURES,
            'sample_size': len(X_train),
            'epochs_trained': len(history.history['loss']),
            'metrics': {
                'accuracy': float(test_accuracy),
                'precision': float(test_precision),
                'recall': float(test_recall),
                'f1': float(f1),
                'loss': float(test_loss)
            },
            'train_time_minutes': train_time / 60,
            'data_was_prescaled': True,  # Important flag!
            'scaler_file': SCALER_FILE
        }
        
        metadata_path = os.path.join(model_dir, "metadata.pkl")
        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        
        # Save training history
        history_path = os.path.join(model_dir, "history.pkl")
        with open(history_path, 'wb') as f:
            pickle.dump(history.history, f)
        
        print(f"   ✅ Model saved: {model_path}")
        print(f"   ✅ Metadata saved: {metadata_path}")
        print(f"   ✅ History saved: {history_path}")

## 9. Results Summary

In [ ]:
print("\n" + "="*80)
print("📊 TRAINING RESULTS SUMMARY")
print("="*80)

summary_df = pd.DataFrame(results).T
summary_df = summary_df.sort_values('f1', ascending=False)

summary_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Loss', 'Train Time (s)', 'Sample Size']

print("\n" + summary_df.to_string())
summary_df

## 10. Visualizations - Training History

In [ ]:
print("\n📊 Creating visualizations...")

# Training history plots
fig, axes = plt.subplots(len(model_builders), 2, figsize=(14, 6*len(model_builders)))
if len(model_builders) == 1:
    axes = axes.reshape(1, -1)

for idx, (model_name, history) in enumerate(histories.items()):
    # Loss
    axes[idx, 0].plot(history.history['loss'], label='Train Loss')
    axes[idx, 0].plot(history.history['val_loss'], label='Val Loss')
    axes[idx, 0].set_title(f'{model_name} - Loss', fontweight='bold')
    axes[idx, 0].set_xlabel('Epoch')
    axes[idx, 0].set_ylabel('Loss')
    axes[idx, 0].legend()
    axes[idx, 0].grid(alpha=0.3)
    
    # Accuracy
    axes[idx, 1].plot(history.history['accuracy'], label='Train Accuracy')
    axes[idx, 1].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[idx, 1].set_title(f'{model_name} - Accuracy', fontweight='bold')
    axes[idx, 1].set_xlabel('Epoch')
    axes[idx, 1].set_ylabel('Accuracy')
    axes[idx, 1].legend()
    axes[idx, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_MODEL_DIR, 'training_history.png'), dpi=300, bbox_inches='tight')
print(f"✅ Saved: {os.path.join(BASE_MODEL_DIR, 'training_history.png')}")
plt.show()

## 11. Visualizations - Performance Comparison

In [ ]:
# Performance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(summary_df))
width = 0.2

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, summary_df[metric], width, label=metric)

axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison (Pre-Scaled Data)', fontweight='bold')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(summary_df.index)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# Training time
axes[1].barh(summary_df.index, summary_df['Train Time (s)'] / 60, color='coral')
axes[1].set_xlabel('Training Time (minutes)')
axes[1].set_title('Training Time Comparison', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_MODEL_DIR, 'model_comparison.png'), dpi=300, bbox_inches='tight')
print(f"✅ Saved: {os.path.join(BASE_MODEL_DIR, 'model_comparison.png')}")
plt.show()

## 12. Final Summary

In [ ]:
print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)

print("\n📊 Models Ranked by F1-Score:")
for idx, (model_name, row) in enumerate(summary_df.iterrows(), 1):
    print(f"   {idx}. {model_name}: {row['F1-Score']:.4f} "
          f"(trained on {int(row['Sample Size']):,} samples in {row['Train Time (s)']/60:.2f} min)")

print(f"\n📁 All models saved to: {BASE_MODEL_DIR}")
print("\n✅ All models trained on PRE-SCALED data!")
print(f"✅ To use on new data, apply the same scaler: {SCALER_FILE}")